In [1]:
import sys
from pathlib import Path
import warnings
from sklearn.exceptions import ConvergenceWarning


ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)


from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.codi.models import CODI


warnings.filterwarnings("ignore", category=ConvergenceWarning)          # sklearn MLP
warnings.filterwarnings("ignore", message="Parameters: {")              # XGBoost unused params
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")


dataset_name = "shuttle"   

dataset_path = ROOT / "raw_data" / f"{dataset_name}.csv"
output_path = ROOT / "discretized_data" / f"{dataset_name}.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Preprocess (same style you used for nursery)
print(f"Discretizing {dataset_path} -> {output_path}")
discretize_preprocess(str(dataset_path), str(output_path))

# Paths for pipeline
input_csv     = str(output_path)
output_dir    = str(ROOT / "sample_data" / dataset_name)
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / dataset_name / "codi")

print("input_csv    :", input_csv)
print("output_dir   :", output_dir)
print("real_test_dir:", real_test_dir)
print("synthetic_dir:", synthetic_dir)

class CoDiCar(CODI):
    def __init__(self):
        super().__init__(
            # Diffusion hyperparameters
            n_steps=50,        # you can increase (e.g. 100) if you want stronger sampling
            beta_1=1e-5,
            beta_T=0.02,

            # Network architecture
            encoder_dim_con=(64, 128, 256),
            encoder_dim_dis=(64, 128, 256),
            nf_con=16,
            nf_dis=64,
            activation="relu",

            # Training hyperparameters
            epochs=30,         # bump up (e.g. 50) if you want more training for car (1781 rows)
            batch_size=512,
            lr_con=2e-3,
            lr_dis=2e-3,
            grad_clip=1.0,

            # Contrastive learning weights
            lambda_con=0.2,
            lambda_dis=0.2,

            # Misc
            random_state=42,
            device=None,       # auto: cuda if available, otherwise cpu
        )

pipeline = TrainTestSplitPipeline(
    model=lambda: CoDiCar()
)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print("\nPipeline result (should include TSTR metrics + result path):")
print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Discretizing C:\Users\Prabu\Downloads\Katabatic\raw_data\shuttle.csv -> C:\Users\Prabu\Downloads\Katabatic\discretized_data\shuttle.csv
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\shuttle.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\shuttle.csv
input_csv    : C:\Users\Prabu\Downloads\Katabatic\discretized_data\shuttle.csv
output_dir   : C:\Users\Prabu\Downloads\Katabatic\sample_data\shuttle
real_test_dir: C:\Users\Prabu\Downloads\Katabatic\sample_data\shuttle
synthetic_dir: C:\Users\Prabu\Downloads\Katabatic\synthetic\shuttle\codi
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
0    0.785970
3    0.153491
4    0.056336
2    0.002953
1    0.000862
6    0.000216
5    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.785948
3    0.153534
4    0.056293


INFO:katabatic.models.codi.models:================================================================================
INFO:katabatic.models.codi.models:Training CoDi Model
INFO:katabatic.models.codi.models:================================================================================
INFO:katabatic.models.codi.models:Loaded training data: (46400, 10)


Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)


INFO:katabatic.models.codi.models:Schema: 0 continuous, 10 categorical columns
INFO:katabatic.models.codi.models:Building models: con_dim=1, cat_dim=89
INFO:katabatic.models.codi.models:Continuous model params: 166,491
INFO:katabatic.models.codi.models:Discrete model params: 368,369
INFO:katabatic.models.codi.models:
Training for 30 epochs...
INFO:katabatic.models.codi.models:Epoch 1/30: loss_con=0.0000, loss_dis=82.9629
INFO:katabatic.models.codi.models:Epoch 5/30: loss_con=0.0000, loss_dis=80.1696
INFO:katabatic.models.codi.models:Epoch 10/30: loss_con=0.0000, loss_dis=79.8754
INFO:katabatic.models.codi.models:Epoch 15/30: loss_con=0.0000, loss_dis=79.7029
INFO:katabatic.models.codi.models:Epoch 20/30: loss_con=0.0000, loss_dis=79.5958
INFO:katabatic.models.codi.models:Epoch 25/30: loss_con=0.0000, loss_dis=79.5149
INFO:katabatic.models.codi.models:Epoch 30/30: loss_con=0.0000, loss_dis=79.4352
INFO:katabatic.models.codi.models:
Generating 46400 synthetic samples...
INFO:katabatic.mo


Results saved to: Results\shuttle\codi_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.8573
F1 Score: 0.8443

MLP:
Accuracy: 0.6370
F1 Score: 0.7013

RF:
Accuracy: 0.9696
F1 Score: 0.9681

XGBoost:
Accuracy: 0.9603
F1 Score: 0.9591

Pipeline result (should include TSTR metrics + result path):
Train test split pipeline executed successfully.
